# 19 — B-Trees & LSM Trees

### Why this exists
Storage engines trade write amplification, read amplification, memory and compaction. This lab makes those costs visible with small deterministic indexes.

### Learning objectives
implement a sorted index and buffered runs; measure lookup work; model compaction

### Prerequisite
Python lists, bisect and complexity reasoning

### Engineering loop
**Predict → Build → Observe → Break → Diagnose → Improve → Generalize → Defend**

## 🖊️ Sketch note
![Sketch note](../assets/sketch-notes/19-b-trees-lsm-trees.svg)

> **Question to carry through the lab:** Why can buffering writes make ingestion cheap while making reads more expensive?

## Capability contract

**Capability:** C05 — Storage Consistency & Distributed Systems  
**Workstream:** Distributed Data Platform  
**Primary roles:** Data Engineer|Platform Engineer|Architect  
**You will prove:** Explain a distributed trade-off


## Mental model
B-trees keep a mutable ordered structure optimized for point/range access. LSM-style designs buffer writes into immutable runs and pay later through reads and compaction.

## 1. Predict before you run
Write down what you expect and what evidence would prove you wrong. Do not change the experiment after seeing the result.

## 2. Build
The first cell creates the smallest system that can answer the question. The later cells change one boundary at a time.

## Agent-native lens

**Agent can do:** routine implementation or a first-pass analysis.

**You must understand:** the mechanism and its assumptions.

**You must verify:** use tests, measurements or adversarial data that could prove the output wrong.

**You must decide:** the trade-off, failure response or production boundary.


## Apprenticeship bridge

Use this notebook to prepare for the following Acme Commerce work scenario(s). The scenario gives you symptoms and constraints; it does **not** give you the diagnosis.

- **S08 — Replication and consistency (DATA-1110)** → [`../work_simulations/08_acme_commerce_replica_lag/README.md`](../work_simulations/08_acme_commerce_replica_lag/README.md)

**How to use the bridge:** finish the experiment and independent challenge here, then enter the ticket without reopening the notebook as an answer key. Bring your own prediction, evidence and verification plan.


In [ ]:
from bisect import bisect_left
class SortedIndex:
    def __init__(self): self.keys=[]
    def put(self,k):
        i=bisect_left(self.keys,k)
        if i==len(self.keys) or self.keys[i]!=k: self.keys.insert(i,k)
    def contains(self,k): return bisect_left(self.keys,k)<len(self.keys) and self.keys[bisect_left(self.keys,k)]==k
idx=SortedIndex()
for k in [9,2,7,4,1,8]: idx.put(k)
print(idx.keys)

In [ ]:
# LSM-like immutable runs.
runs=[[1,4,9],[2,7],[3,8],[5,6]]
def read_cost(key,runs):
    checks=0
    for run in reversed(runs):
        checks+=1
        if key in run: return True,checks
    return False,checks
print({k:read_cost(k,runs)[1] for k in [1,5,10]})

In [ ]:
def compact(a,b): return sorted(set(a)|set(b))
compacted=[compact(runs[i],runs[i+1]) for i in range(0,len(runs),2)]
print("runs before",len(runs),"after one compaction round",len(compacted),compacted)

In [ ]:
# Break: add many tiny runs.
hot_runs=[[i] for i in range(20)]
print("read checks for oldest key:",read_cost(0,hot_runs)[1])
print("read checks for absent key:",read_cost(999,hot_runs)[1])

## 3. Measure
Do not stop at “it worked”. Record a correctness signal, a failure signal, and at least one quantitative measurement where the concept permits it.

## 4. Break deliberately
Introduce one controlled failure. Keep the failure reproducible so another engineer can reproduce the observation.

## 5. Diagnose
Use evidence, not the implementation you expected. State: **symptom → mechanism → evidence → boundary**. If two explanations fit, design one more measurement.

## 6. Improve
Make the smallest change that addresses the mechanism. Re-run the original experiment and the failure case.

## 7. Generalize
Ask what changes at 10× load, with retries, multiple writers/readers, partial failure, or changing contracts. Separate what this toy proves from what it only illustrates.

## 8. Production bridge
Production storage engines choose structures from workload shape: write rate, point reads, range scans, durability, compaction budget and memory.

## 9. Independent challenge
Model tombstones and overwrites, then compare read amplification before and after compaction. Explain when compaction can become the bottleneck.

### Evidence to keep
sorted index behavior, read-check counts, compaction reduction, and a read/write/space trade-off table

## 10. Explain it in 60–90 seconds
Explain the mechanism, your measurement, the failure you injected, the diagnosis, and the production decision you would make. Avoid tool names until the mental model is clear.